# 📄 Module 2: Content-Based Recommendation with Explainable AI (XAI)
**AI-Powered Personalized Product Recommendation & Analytics Platform**

In this notebook:
1. Feature Preprocessing & Metadata Soup Construction
2. TF-IDF (Term Frequency-Inverse Document Frequency) Vectorization
3. Pairwise Cosine Similarity Matrix Computation
4. Product Similarity Retrieval (e.g. iPhone 15 recommendations)
5. Explainable AI (XAI) Feature & Smart Budget Alternatives Generation


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

BASE_DIR = '..'
products_path = os.path.join(BASE_DIR, 'data', 'processed', 'clean_products.csv')
df_products = pd.read_csv(products_path)

print(f"Loaded {len(df_products)} processed products.")


### 1. Content Soup Inspection


In [ ]:
# Inspect sample content soup used for vectorization
print("Sample Content Soup:")
print(df_products[['product_name', 'content_soup']].iloc[0].to_dict())


### 2. TF-IDF Vectorization


In [ ]:
vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    max_features=3000,
    sublinear_tf=True
)
tfidf_matrix = vectorizer.fit_transform(df_products['content_soup'])
print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")
print(f"Extracted {len(vectorizer.get_feature_names_out())} vocabulary features.")


### 3. Cosine Similarity Computation


In [ ]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"Cosine Similarity Matrix Shape: {cosine_sim.shape}")
print(f"Min Similarity: {cosine_sim.min():.3f}, Max Similarity: {cosine_sim.max():.3f}")


### 4. Similar Products Retrieval


In [ ]:
def get_similar_products(product_id, top_n=5):
    idx = df_products[df_products['product_id'] == product_id].index[0]
    scores = list(enumerate(cosine_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    # Exclude self
    scores = [s for s in scores if s[0] != idx][:top_n]
    
    recs = []
    for i, score in scores:
        p = df_products.iloc[i]
        recs.append({
            'Product ID': p['product_id'],
            'Product Name': p['product_name'],
            'Category': p['category'],
            'Brand': p['brand'],
            'Price (₹)': p['price'],
            'Similarity Score': round(float(score), 4)
        })
    return pd.DataFrame(recs)

# Test with iPhone 15
print("Similar items to Apple iPhone 15:")
get_similar_products('PROD_ELEC_001', top_n=5)


### 5. Smart Budget Alternatives Feature


In [ ]:
def get_smart_alternatives(product_id, top_n=3):
    idx = df_products[df_products['product_id'] == product_id].index[0]
    base = df_products.iloc[idx]
    base_price = base['price']
    
    candidates = []
    for i, row in df_products.iterrows():
        if i == idx:
            continue
        if row['price'] < base_price and row['category'] == base['category']:
            sim = cosine_sim[idx][i]
            savings = round(((base_price - row['price']) / base_price) * 100, 1)
            candidates.append({
                'Product Name': row['product_name'],
                'Price (₹)': row['price'],
                'Savings (%)': savings,
                'Similarity Score': round(float(sim), 4),
                'Explanation': f"Save {savings}% (₹{int(base_price - row['price']):,}) with similar features in {row['subcategory']}."
            })
    return pd.DataFrame(candidates).sort_values(by='Similarity Score', ascending=False).head(top_n)

print("Smart Alternatives for Apple iPhone 15:")
get_smart_alternatives('PROD_ELEC_001', top_n=3)
